## 🔍 在 `gensim` 的 LDA 中，`eta` 是怎麼用的？

### 📌 `eta` 是每個主題對每個詞的 **先驗分佈參數（Dirichlet prior）**

數學上，它是 LDA 中：

$$
\beta_k \sim \text{Dirichlet}(\eta_k)
$$

其中：

* $\beta_k$ 是 topic $k$ 的詞彙分佈（K × V）
* $\eta_k$ 是第 $k$ 個 topic 的 Dirichlet 分布參數向量（維度 V）

---

## ✅ 具體來說：

| 你設定的 η 值    | 影響的是什麼？                   |
| ----------- | ------------------------- |
| η\[k]\[v] 高 | 詞 v 出現在 topic k 的機率先驗變大   |
| η\[k]\[v] 低 | 詞 v 出現在 topic k 的機率先驗趨近 0 |

而且：

* **η 不是乘進去也不是加進去**
* 它是 **Dirichlet 分佈的參數 α 一樣的東西，但是針對詞（word）而非 topic**

---

## 📦 在 `gensim.LdaModel` 中：

當你這樣設定：

```python
LdaModel(..., eta=your_eta_matrix, ...)
```

gensim 做的是：

* 把 `eta[k][v]` 當作 **topic k 下，詞 v 的機率的先驗參數**
* 在 Gibbs sampling / Variational Inference 過程中，這個先驗會影響 $\phi_{k,v}$ 的學習方向與穩定性

---

## 📊 例子：如果你給

| topic | token  | η值   | 代表意義                      |
| ----- | ------ | ---- | ------------------------- |
| 0     | "tech" | 0.5  | 希望在 topic 0 中，"tech" 出現較多 |
| 1     | "tech" | 0.01 | 希望 topic 1 中幾乎不會選到這個詞     |

這代表：

> 在沒有任何觀察資料下，LDA 會傾向把 "tech" 放進 topic 0，而非 topic 1。

---

## ⚠️ 重點提醒：**η 的效果是偏好，不是硬限制！**

* 如果某個 token 在文件中反覆與某個主題共現，LDA 還是會調整回來
* 但 η 高會讓其「比較容易被選進該 topic」
* 所以你現在做的：**用 GAP 結果提升 η 的方式，是一種 soft guidance**

---

## ✅ 總結你該記得的

| 問題                  | 答案                                 |
| ------------------- | ---------------------------------- |
| gensim 的 `eta` 是什麼？ | 每個主題的詞彙分佈先驗參數（Dirichlet）           |
| 是怎麼用的？              | 當作 sampling / inference 時的 prior   |
| 高 η 是什麼意思？          | 越偏好這個詞在該 topic 出現                  |
| 是相乘還是相加？            | 都不是，是做為 Dirichlet 的 α 參數用於抽樣機率分佈之用 |
| 會強制嗎？               | 不會，只是增加傾向（除非 η 非常極端）               |

---

## 📄 1. LDA 模型中的 η 是什麼？

在原文中（LDA paper, Section 2），topic 的詞分佈被建模為：

$$
\beta_k \sim \text{Dirichlet}(\eta)
$$

其中：

* $\beta_k$ 是第 $k$ 個主題的詞機率分佈（對 vocabulary 做 softmax）
* $\eta$ 是一個長度為 $V$ 的向量，是 Dirichlet 分布的參數

### ❗ 關鍵：**不是單一 η 值，而是對所有詞的 η 向量**

> 在原始論文中，LDA 假設所有 topic 共用相同的 η，且 η 通常為常數（symmetric Dirichlet）

---

## 📊 2. Dirichlet 的作用（如何影響分類）

Dirichlet 是共軛先驗分布，用來產生 softmax 的分佈。具體來說：

若：

$$
\beta_k \sim \text{Dirichlet}(\eta)
$$

則每個詞 $w_i$ 在 topic k 中的機率為：

$$
\mathbb{E}[\beta_{k,i}] = \frac{\eta_i}{\sum_{j=1}^{V} \eta_j}
$$

所以：

* $\eta_i$ 越大 → 詞 i 越可能在 topic k 中被選到
* 所以 **η 的數值是加法性的（加進 Dirichlet 的 α 向量）**

---

## 🔁 與你問的「加還是乘」有什麼關係？

### ✅ 回答：

* 在 **數學建模中是「加進 Dirichlet 分布參數」**，不是在 posterior 裡相乘或相加。
* 在 **變異推論（variational inference）或 Gibbs Sampling** 中，它會**加進 word-topic 分配的先驗中**，導致在迭代中更傾向於某些主題。

---

## 📌 原文說明位置

你可以在原文：

📄 Blei et al. (2003), *Latent Dirichlet Allocation*

* Section 2.1 Generative Process
* Section 3.1 Variational Inference
* Appendix A.3（Variational EM）
  中找到 η 的使用與影響方式。

---

## 🔍 關鍵公式（變異 EM 中）

E-step 中的變異參數 $\phi_{n,k}$ 更新式包含：

$$
\phi_{n,k} \propto \exp \left( \mathbb{E}_q[\log \theta_k] + \mathbb{E}_q[\log \beta_{k,w_n}] \right)
$$

而其中：

$$
\mathbb{E}_q[\log \beta_{k,w_n}] = \Psi(\eta_{w_n} + N_{kw_n}) - \Psi\left(\sum_v (\eta_v + N_{kv})\right)
$$

其中 $\Psi$ 是 digamma 函數、$N_{kw_n}$ 是 counts，所以：

* **η 是直接加進機率估計中的 numerators**，確實是「加」不是乘

---

## ✅ 結論整理

| 問題                            | 解答                                                        |
| ----------------------------- | --------------------------------------------------------- |
| LDA 原文有定義 η 嗎？                | ✅ 有，在 generative process 中直接定義每個 topic 的詞分佈為 Dirichlet(η) |
| η 對分類的影響是什麼？                  | η 決定了 topic 中每個詞出現的 prior 機率分佈，大 η 的詞在該 topic 中預期機率高      |
| 是用「加」還是「乘」的？                  | ✅ 是 **加法性**：η 是加進 Dirichlet α vector 裡，不是乘進機率或似然中         |
| η 對 Gibbs 或 Variational 有影響嗎？ | ✅ 有，它影響 E-step 的更新公式（進 digamma）                           |